In [1]:
import sys; print(sys.executable)

C:\venvs\guardianshield-clv\Scripts\python.exe


# Engagement 2 — Notebook 02a: BG/NBD Baseline (Negative Result)

## Status: **DEPRECATED**

This notebook documents an **abandoned modelling approach**.

## Why it was abandoned

BG/NBD is designed for continuous-time purchase behaviour (e-commerce, retail). Our insurance data is **contractual and annual**: customers renew once per policy year, at a fixed date.

Attempting to fit BG/NBD produced **degenerate parameters**:
- `a ≈ 0`, `b ≈ 0` across multiple penalizer settings (0.001, 0.1, 0.5)
- These values indicate the model detected **no dropout signal**, because renewals happen on schedule for all active customers

## What we did instead

Switched to **survival analysis** (Kaplan-Meier + Cox Proportional Hazards) — the correct framework for contractual/subscription data.

See `02b_clv_survival.ipynb`.

## Value of this notebook

Documented negative results demonstrate **model selection judgement** — the ability to recognise when a popular tool doesn't fit the problem, and to choose the right one instead.

In [2]:
from pathlib import Path
print(Path.cwd())

C:\Users\adebi\OneDrive\Desktop\DataScienceConsultingPortfolio\01_Portfolio-Projects\OkYeahInsight-Analytics\GuardianShield-Insurance\Engagement-2-Customer-Lifetime-Value


# Engagement 2 — CLV Modelling (BG/NBD + Gamma-Gamma)

**Client:** GuardianShield Insurance (fictional)
**Engagement:** Customer Lifetime Value Prediction
**Notebook:** 02 — Probabilistic CLV Model
**Author:** Yinka Adebimpe
**Date:** [Today's date]

## Purpose

Predict 12-month Customer Lifetime Value (CLV) per customer using:

1. **BG/NBD** — models transaction frequency and silent churn
2. **Gamma-Gamma** — models expected monetary value per transaction

## Method summary

- Transaction event = **renewal** (not payment), removing payment-cadence bias
- Snapshot date: **2025-12-29**
- Horizon: **12 months forward**
- Customers with zero renewals are excluded from CLV modelling (insufficient history) but remain in the RFM table

## Output

A `clv.clv_predictions` table with per-customer:
- Probability alive
- Expected renewals in next 12 months
- Expected monetary value per renewal
- **Predicted 12-month CLV (£)**

In [3]:
import sys
from pathlib import Path

# Find the project root by walking up from cwd until we find src/config.py
def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "config.py").exists():
            return p
    raise FileNotFoundError("Could not find project root (no src/config.py found)")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

# Now the rest of the imports work regardless of where the notebook lives
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

from lifetimes import BetaGeoFitter, GammaGammaFitter

from config import DB_URL, DB_SCHEMA, IMAGES_DIR, PROCESSED_DATA_DIR

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

engine = create_engine(DB_URL)

print("Setup complete.")
print(f"Python: {sys.executable}")
print(f"lifetimes: imported successfully")

Project root: C:\Users\adebi\OneDrive\Desktop\DataScienceConsultingPortfolio\01_Portfolio-Projects\OkYeahInsight-Analytics\GuardianShield-Insurance\Engagement-2-Customer-Lifetime-Value
Setup complete.
Python: C:\venvs\guardianshield-clv\Scripts\python.exe
lifetimes: imported successfully


In [4]:
# ============================================
# LOAD DATA
# ============================================

transactions = pd.read_sql(
    f"SELECT * FROM {DB_SCHEMA}.transactions",
    engine,
    parse_dates=["transaction_date"],
)

customers = pd.read_sql(
    f"SELECT * FROM {DB_SCHEMA}.customers",
    engine,
    parse_dates=["policy_start_date", "churned_date"],
)

snapshot_date = transactions["transaction_date"].max() + pd.Timedelta(days=1)

print(f"Transactions: {len(transactions):,}")
print(f"Customers:    {len(customers):,}")
print(f"Snapshot:     {snapshot_date.date()}")

Transactions: 107,884
Customers:    5,000
Snapshot:     2025-12-29


C:\Users\adebi\AppData\Local\Temp\ipykernel_16460\2164844505.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  snapshot_date = transactions["transaction_date"].max() + pd.Timedelta(days=1)


In [5]:
# ============================================
# BUILD LIFETIMES INPUT
# ============================================
# For BG/NBD we model RENEWAL events.
# The initial policy purchase is the "first transaction" (implicit, always 1).
# Each renewal is a "repeat transaction".
#
# lifetimes inputs per customer:
#   frequency       = number of repeat events (renewals)
#   recency         = days between first purchase and last renewal
#   T               = days between first purchase and snapshot
#   monetary_value  = AVERAGE ANNUAL PREMIUM (£ per policy-year)
#                     — normalises monthly vs annual payment structures

# Renewals only
renewals = transactions[transactions["renewed"] == True].copy()

# First purchase date per customer
first_purchase = (
    customers[["customer_id", "policy_start_date"]]
    .set_index("customer_id")["policy_start_date"]
)

# Renewal stats
renewal_stats = renewals.groupby("customer_id").agg(
    last_renewal=("transaction_date", "max"),
    frequency=("transaction_id", "count"),
)

# --- Average annual premium ---
# Total premium paid per customer
total_premium = (
    transactions
    .groupby("customer_id")["premium_amount"]
    .sum()
)

# Tenure in years (from first to last transaction)
tx_dates = transactions.groupby("customer_id")["transaction_date"].agg(["min", "max"])
tenure_years = (tx_dates["max"] - tx_dates["min"]).dt.days / 365.25
tenure_years = tenure_years.clip(lower=0.5)   # at least 6 months to avoid extreme ratios

annual_premium = (total_premium / tenure_years).rename("monetary_value")

# Combine
lf = renewal_stats.join(first_purchase).join(annual_premium)

lf["recency"] = (lf["last_renewal"] - lf["policy_start_date"]).dt.days
lf["T"] = (snapshot_date - lf["policy_start_date"]).dt.days

# Keep only customers with at least 1 renewal (Option A)
lf = lf[lf["frequency"] >= 1].copy()

# Keep only needed columns
lf = lf[["frequency", "recency", "T", "monetary_value"]]

print(f"lifetimes input shape: {lf.shape}")
print(f"Customers included: {len(lf):,}")
print(f"Customers excluded (0 renewals): {5_000 - len(lf):,}")
print(f"\nSample:")
print(lf.head(10))
print(f"\nSummary (annual premium basis):")
print(lf.describe().round(2))

lifetimes input shape: (3383, 4)
Customers included: 3,383
Customers excluded (0 renewals): 1,617

Sample:
             frequency  recency     T  monetary_value
customer_id                                          
GS-000001            2      731  1149      783.500690
GS-000002            2      731  1049      311.310276
GS-000004            4     1461  1802      893.832500
GS-000006            3     1096  1444      486.819503
GS-000007            4     1461  1725     1012.702218
GS-000008            3     1096  1549      856.686090
GS-000009            3     1095  2002      355.785660
GS-000010            3     1096  1409      850.829213
GS-000011            1      363   486     1668.790021
GS-000012            3     1096  1204      504.552719

Summary (annual premium basis):
       frequency  recency        T  monetary_value
count    3383.00  3383.00  3383.00         3383.00
mean        2.49   910.31  1194.73          841.63
std         1.26   461.65   472.07          426.58
min     

In [6]:
# ============================================
# SANITY CHECK
# ============================================

bad_rows = lf[lf["recency"] > lf["T"]]
if len(bad_rows) > 0:
    print(f"WARNING: {len(bad_rows)} rows have recency > T")
    print(bad_rows.head())
else:
    print("✓ All rows satisfy recency <= T")

print(f"\nFrequency distribution:")
print(lf["frequency"].value_counts().sort_index())

print(f"\nMonetary value stats (annual premium):")
print(lf["monetary_value"].describe().round(2))

✓ All rows satisfy recency <= T

Frequency distribution:
frequency
1    984
2    797
3    801
4    554
5    247
Name: count, dtype: int64

Monetary value stats (annual premium):
count    3383.00
mean      841.63
std       426.58
min       142.42
25%       547.78
50%       756.34
75%      1021.08
max      4947.05
Name: monetary_value, dtype: float64


In [7]:
# ============================================
# FIT BG/NBD MODEL
# ============================================

bgf = BetaGeoFitter(penalizer_coef=0.5)
bgf.fit(frequency=lf["frequency"], recency=lf["recency"], T=lf["T"])
print(bgf.params_)

r        6.594766e-01
alpha    3.029164e+02
a        8.942975e-16
b        4.862237e-06
dtype: float64


In [8]:
bgf = BetaGeoFitter(penalizer_coef=0.1)   # was 0.001
bgf.fit(frequency=lf["frequency"], recency=lf["recency"], T=lf["T"])
print(bgf.params_)

r        1.266931e+00
alpha    5.956054e+02
a        1.654009e-17
b        9.879126e-07
dtype: float64


In [9]:
bgf = BetaGeoFitter(penalizer_coef=0.001)

bgf.fit(
    frequency=lf["frequency"],
    recency=lf["recency"],
    T=lf["T"],
)

print("BG/NBD model fitted.")
print(f"\nParameters:")
for name, value in bgf.params_.items():
    print(f"  {name}: {value:.4f}")

BG/NBD model fitted.

Parameters:
  r: 7.1175
  alpha: 3403.5302
  a: 0.0000
  b: 0.0000


C:\venvs\guardianshield-clv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
